# Comparison among EFA, PCA, KPCA, and Autoencoder

This notebook compares four dimensionality-reduction / representation-learning methods for CBCL questionnaire data:

- Exploratory Factor Analysis (EFA)
- Principal Component Analysis (PCA)
- Kernel Principal Component Analysis (KPCA)
- Autoencoder

The analysis uses encoding dimensions 3, 4, and 5. For each method and dimension, the notebook calculates latent factors and reconstruction-based evaluation metrics when available.

KPCA includes two variants:

- Linear KPCA: full-data latent factors, reconstruction MSE, and total EVR.
- RBF KPCA: nested cross-validated reconstruction EVR.

The notebook also keeps PCA outer-CV EVR and Autoencoder held-out test-set reconstruction results as additional evaluation outputs.

## Metric sources

- `full_data_reconstruction`: model fit and reconstruction on the full standardized matrix.
- `outer_cv_reconstruction_evr`: reconstruction EVR estimated by outer cross-validation.
- `nested_cv_reconstruction_evr`: reconstruction EVR estimated by nested cross-validation.
- `test_set_reconstruction`: reconstruction metrics evaluated on the held-out Autoencoder test set.


## 1. Configuration and paths

In [ ]:

import os
from pathlib import Path

SEED = 52
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

CODE_DIR = Path.cwd()
DATA_DIR = CODE_DIR.parent / 'data'
DATA_FILE = DATA_DIR / 'cbcl_data_remove_low_frequency.csv'
OUTPUT_DIR = CODE_DIR.parent / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DIMENSIONS = [3, 4, 5]

print(f'Code directory: {CODE_DIR}')
print(f'Data file: {DATA_FILE}')
print(f'Output directory: {OUTPUT_DIR}')


## 2. Imports and reproducibility settings

In [ ]:

import random
import warnings
from typing import Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st
import torch
from factor_analyzer import FactorAnalyzer
from IPython.display import display
from sklearn.decomposition import KernelPCA, PCA
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from model_code import AE, _total_evr_from_recon, set_deterministic

warnings.filterwarnings('default')


def set_global_reproducibility(seed: int) -> None:
    """Set seeds consistently because this notebook compares deterministic and stochastic methods."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    set_deterministic(seed)


set_global_reproducibility(SEED)


## 3. Data reading and preprocessing

In [ ]:


def load_cbcl_data(data_file: Path) -> Tuple[pd.DataFrame, np.ndarray]:
    """Load the original CBCL file and preserve the original column selection logic."""
    if not data_file.exists():
        raise FileNotFoundError(f'Could not find {data_file}')

    qns = pd.read_csv(data_file, encoding='utf-8')
    X = qns.iloc[:, 1:].values
    return qns, X


def split_and_standardize(X: np.ndarray, seed: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Use the original 80/10/10 split and fit the scaler on training data only."""
    X_train_raw, X_temp = train_test_split(X, test_size=0.2, random_state=seed)
    X_val_raw, X_test_raw = train_test_split(X_temp, test_size=0.5, random_state=seed)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    X_test = scaler.transform(X_test_raw)
    X_scaled = scaler.fit_transform(X)
    return X_train, X_val, X_test, X_scaled


qns, X = load_cbcl_data(DATA_FILE)
X_train, X_val, X_test, X_scaled = split_and_standardize(X, SEED)

print(f'Questionnaire table shape: {qns.shape}')
print(f'Feature matrix shape: {X.shape}')
print(f'Train shape: {X_train.shape}, validation shape: {X_val.shape}, test shape: {X_test.shape}')


## 4. Helper functions

In [ ]:

ResultDict = Dict[str, Dict[str, object]]
AnalysisResults = Dict[str, Dict[str, object]]


def make_dim_key(encoding_dim: int) -> str:
    return f'dim{encoding_dim}'


def reconstruction_mse(X_true: np.ndarray, X_reconstructed: np.ndarray) -> float:
    return float(np.mean((X_true - X_reconstructed) ** 2))


def build_result_entry(
    latent_factors: np.ndarray | None,
    reconstructed: np.ndarray | None,
    mse: float | None,
    evr: float,
    evr_std: float | None = None,
    extra: Dict[str, object] | None = None,
) -> Dict[str, object]:
    entry = {
        'latent_factors': latent_factors,
        'reconstructed': reconstructed,
        'mse': np.nan if mse is None else float(mse),
        'evr': float(evr),
        'evr_std': np.nan if evr_std is None else float(evr_std),
    }
    if extra:
        entry.update(extra)
    return entry


def save_latent_factors(
    output_dir: Path,
    method_name: str,
    results_dict: ResultDict,
    dimensions: Iterable[int],
) -> None:
    """Save latent factors only for analyses that actually produce a full-data latent representation."""
    method_slug = method_name.lower().replace(' ', '_')

    for encoding_dim in dimensions:
        dim_key = make_dim_key(encoding_dim)
        latent_factors = results_dict[dim_key].get('latent_factors')
        if latent_factors is None:
            continue

        latent_df = pd.DataFrame(
            latent_factors,
            columns=[f'factor_{i}' for i in range(1, encoding_dim + 1)],
        )
        output_file = output_dir / f'latent_factors_{method_slug}_dim{encoding_dim}.csv'
        latent_df.to_csv(output_file, index=False)


def build_analysis_results() -> AnalysisResults:
    return {
        'EFA': {
            'family': 'EFA',
            'variant': 'default',
            'evaluation': 'full_data_reconstruction',
            'results': None,
        },
        'PCA': {
            'family': 'PCA',
            'variant': 'default',
            'evaluation': 'full_data_reconstruction',
            'results': None,
        },
        'PCA_outer_cv': {
            'family': 'PCA',
            'variant': 'default',
            'evaluation': 'outer_cv_reconstruction_evr',
            'results': None,
        },
        'KPCA_linear': {
            'family': 'KPCA',
            'variant': 'linear',
            'evaluation': 'full_data_reconstruction',
            'results': None,
        },
        'KPCA_rbf_nested_cv': {
            'family': 'KPCA',
            'variant': 'rbf',
            'evaluation': 'nested_cv_reconstruction_evr',
            'results': None,
        },
        'Autoencoder': {
            'family': 'Autoencoder',
            'variant': 'default',
            'evaluation': 'full_data_reconstruction',
            'results': None,
        },
        'Autoencoder_test': {
            'family': 'Autoencoder',
            'variant': 'default',
            'evaluation': 'test_set_reconstruction',
            'results': None,
        },
    }


def build_comparison_table(analysis_results: AnalysisResults, dimensions: Iterable[int]) -> pd.DataFrame:
    records: List[Dict[str, object]] = []

    for analysis_key, metadata in analysis_results.items():
        results_dict = metadata['results']
        if results_dict is None:
            continue

        for encoding_dim in dimensions:
            dim_key = make_dim_key(encoding_dim)
            result = results_dict[dim_key]
            records.append({
                'analysis_key': analysis_key,
                'method_family': metadata['family'],
                'method_variant': metadata['variant'],
                'evaluation': metadata['evaluation'],
                'display_name': f"{metadata['family']} ({metadata['variant']})" if metadata['variant'] != 'default' else metadata['family'],
                'dim': encoding_dim,
                'mse': result.get('mse', np.nan),
                'evr': result.get('evr', np.nan),
                'evr_std': result.get('evr_std', np.nan),
                'gamma0': result.get('gamma0', np.nan),
            })

    comparison_df = pd.DataFrame(records)
    comparison_df = comparison_df.sort_values(['method_family', 'method_variant', 'evaluation', 'dim']).reset_index(drop=True)
    return comparison_df


def plot_metric_line(
    comparison_df: pd.DataFrame,
    metric: str,
    title: str,
    ylabel: str,
    output_file: Path,
    allowed_evaluations: Iterable[str] | None = None,
) -> None:
    plot_df = comparison_df.copy()
    if allowed_evaluations is not None:
        plot_df = plot_df[plot_df['evaluation'].isin(list(allowed_evaluations))]

    plt.figure(figsize=(8, 5))
    for display_name, df_model in plot_df.groupby('display_name'):
        df_model = df_model.sort_values('dim')
        y = pd.to_numeric(df_model[metric], errors='coerce')
        if y.notna().sum() == 0:
            continue
        plt.plot(df_model['dim'], y, marker='o', linestyle='-', label=display_name)

    plt.xlabel('Encoding Dimension')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(list(DIMENSIONS))
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()


def plot_evr_grouped_bar(
    comparison_df: pd.DataFrame,
    output_file: Path,
    allowed_evaluations: Iterable[str] | None = None,
) -> None:
    plot_df = comparison_df.copy()
    if allowed_evaluations is not None:
        plot_df = plot_df[plot_df['evaluation'].isin(list(allowed_evaluations))]

    plot_df = plot_df.copy()
    plot_df['label'] = plot_df.apply(
        lambda row: row['display_name'] if row['evaluation'] == 'full_data_reconstruction' else f"{row['display_name']} [{row['evaluation']}]",
        axis=1,
    )

    dims = sorted(plot_df['dim'].unique())
    labels = list(plot_df['label'].unique())
    x = np.arange(len(dims))
    width = 0.8 / max(len(labels), 1)

    plt.figure(figsize=(10, 5))
    for i, label in enumerate(labels):
        df_label = plot_df[plot_df['label'] == label].set_index('dim').reindex(dims)
        y = pd.to_numeric(df_label['evr'], errors='coerce').values
        plt.bar(x + (i - (len(labels) - 1) / 2) * width, y, width=width, label=label)

    plt.xlabel('Encoding Dimension')
    plt.ylabel('Total Explained Variance')
    plt.title('Total Explained Variance by Method and Dimension')
    plt.xticks(x, dims)
    plt.grid(True, axis='y', alpha=0.3)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()


## 5. Full-data EFA, PCA, and linear KPCA

In [ ]:


def run_full_data_linear_methods(X_scaled: np.ndarray, dimensions: Iterable[int]) -> Tuple[ResultDict, ResultDict, ResultDict]:
    """Run the original full-data EFA, PCA, and linear KPCA analyses."""
    results_efa: ResultDict = {}
    results_pca: ResultDict = {}
    results_kpca_linear: ResultDict = {}

    for encoding_dim in dimensions:
        dim_key = make_dim_key(encoding_dim)
        print(f'\n=== Encoding Dim: {encoding_dim} ===')

        fa = FactorAnalyzer(n_factors=encoding_dim, rotation='geomin_obl')
        fa.fit(X_scaled)
        efa_scores = fa.transform(X_scaled)
        X_recon_efa = efa_scores @ fa.loadings_.T
        mse_efa = reconstruction_mse(X_scaled, X_recon_efa)
        evr_efa = _total_evr_from_recon(X_scaled, X_recon_efa)
        print(f'EFA -> MSE={mse_efa:.6f}, Total EVR={evr_efa:.4f}')
        results_efa[dim_key] = build_result_entry(
            latent_factors=efa_scores,
            reconstructed=X_recon_efa,
            mse=mse_efa,
            evr=evr_efa,
        )

        pca = PCA(n_components=encoding_dim)
        pca_scores = pca.fit_transform(X_scaled)
        X_recon_pca = pca.inverse_transform(pca_scores)
        mse_pca = reconstruction_mse(X_scaled, X_recon_pca)
        evr_pca = _total_evr_from_recon(X_scaled, X_recon_pca)
        print(f'PCA -> MSE={mse_pca:.6f}, Total EVR={evr_pca:.4f}')
        results_pca[dim_key] = build_result_entry(
            latent_factors=pca_scores,
            reconstructed=X_recon_pca,
            mse=mse_pca,
            evr=evr_pca,
        )

        kpca = KernelPCA(
            n_components=encoding_dim,
            kernel='linear',
            fit_inverse_transform=True,
        )
        kpca_scores = kpca.fit_transform(X_scaled)
        X_recon_kpca = kpca.inverse_transform(kpca_scores)
        mse_kpca = reconstruction_mse(X_scaled, X_recon_kpca)
        evr_kpca = _total_evr_from_recon(X_scaled, X_recon_kpca)
        print(f'Linear KPCA -> MSE={mse_kpca:.6f}, Total EVR={evr_kpca:.4f}')
        results_kpca_linear[dim_key] = build_result_entry(
            latent_factors=kpca_scores,
            reconstructed=X_recon_kpca,
            mse=mse_kpca,
            evr=evr_kpca,
        )

    return results_efa, results_pca, results_kpca_linear


results_efa, results_pca, results_kpca_linear = run_full_data_linear_methods(X_scaled, DIMENSIONS)


## 6. PCA outer-CV EVR and RBF KPCA nested-CV EVR

In [ ]:


def recon_evr_from_estimator(estimator, X, y=None) -> float:
    """Compute reconstruction EVR for estimators that support transform and inverse_transform."""
    latent = estimator.transform(X)
    X_reconstructed = estimator.inverse_transform(latent)

    X = np.asarray(X)
    X_reconstructed = np.asarray(X_reconstructed)
    ss_total = np.sum((X - X.mean(axis=0)) ** 2)
    ss_resid = np.sum((X - X_reconstructed) ** 2)
    return float(1.0 - ss_resid / ss_total)


def median_heuristic_gamma(X: np.ndarray, n_samples: int = 2000, random_state: int = 0) -> float:
    """Use the median pairwise distance to center the RBF gamma search range."""
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    idx = rng.choice(n, size=min(n, n_samples), replace=False)
    Xs = X[idx]
    distances = pairwise_distances(Xs, metric='euclidean')
    median_distance = np.median(distances[distances > 0])
    sigma2 = median_distance ** 2
    return 1.0 / (2.0 * sigma2 + 1e-12)


def run_pca_and_rbf_kpca_cv(
    X_raw: np.ndarray,
    dimensions: Iterable[int],
    seed: int,
) -> Tuple[ResultDict, ResultDict]:
    """Retain the original cross-validated PCA and nested-CV RBF KPCA reconstruction EVR analysis."""
    outer_cv = KFold(n_splits=5, shuffle=True, random_state=seed)
    gamma0 = median_heuristic_gamma(X_raw, random_state=seed)

    results_pca_cv: ResultDict = {}
    results_kpca_rbf_cv: ResultDict = {}

    for encoding_dim in dimensions:
        dim_key = make_dim_key(encoding_dim)
        print(f'\n=== n_components = {encoding_dim} ===')

        kpca_pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('kpca', KernelPCA(
                n_components=encoding_dim,
                kernel='rbf',
                fit_inverse_transform=True,
                eigen_solver='auto',
                random_state=seed,
            )),
        ])

        param_dist = {
            'kpca__gamma': st.loguniform(gamma0 / 100, gamma0 * 100),
            'kpca__alpha': st.loguniform(1e-6, 1e-1),
        }

        kpca_search = RandomizedSearchCV(
            kpca_pipe,
            param_distributions=param_dist,
            n_iter=30,
            scoring=recon_evr_from_estimator,
            cv=3,
            random_state=seed,
            error_score='raise',
        )

        pca_pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=encoding_dim, random_state=seed)),
        ])

        pca_scores = cross_val_score(
            pca_pipe,
            X_raw,
            y=None,
            scoring=recon_evr_from_estimator,
            cv=outer_cv,
        )

        kpca_nested_scores = cross_val_score(
            kpca_search,
            X_raw,
            y=None,
            scoring=recon_evr_from_estimator,
            cv=outer_cv,
        )

        print(f'PCA outer CV EVR: mean={pca_scores.mean():.4f}, std={pca_scores.std():.4f}')
        print(f'RBF KPCA nested CV EVR: mean={kpca_nested_scores.mean():.4f}, std={kpca_nested_scores.std():.4f}')

        results_pca_cv[dim_key] = build_result_entry(
            latent_factors=None,
            reconstructed=None,
            mse=None,
            evr=float(pca_scores.mean()),
            evr_std=float(pca_scores.std()),
        )

        results_kpca_rbf_cv[dim_key] = build_result_entry(
            latent_factors=None,
            reconstructed=None,
            mse=None,
            evr=float(kpca_nested_scores.mean()),
            evr_std=float(kpca_nested_scores.std()),
            extra={'gamma0': float(gamma0)},
        )

    return results_pca_cv, results_kpca_rbf_cv


results_pca_cv, results_kpca_rbf_cv = run_pca_and_rbf_kpca_cv(X, DIMENSIONS, SEED)


## 7. Autoencoder

In [ ]:


def run_autoencoder_analysis(
    X_train: np.ndarray,
    X_val: np.ndarray,
    X_test: np.ndarray,
    X_scaled: np.ndarray,
    dimensions: Iterable[int],
    seed: int,
) -> Tuple[ResultDict, ResultDict]:
    """Train and evaluate the original Autoencoder configuration for each latent dimension."""
    results_ae_test: ResultDict = {}
    results_ae_full: ResultDict = {}

    for encoding_dim in dimensions:
        dim_key = make_dim_key(encoding_dim)
        print(f'\n=== Encoding Dim: {encoding_dim} ===')
        set_global_reproducibility(seed)

        model = AE(
            X_train,
            X_val,
            encoding_dim=encoding_dim,
            h1=192,
            h2=64,
            seed=seed,
            lr=0.003,
            lambda_decorr=0.005,
            weight_decay=8e-05,
        )
        model.train(show_plot=True)

        latent_test, _, _, evr_test, reconstructed_test, _ = model.evaluate_on_data(X_test)
        mse_test = reconstruction_mse(X_test, reconstructed_test)
        print(f'Autoencoder test set -> MSE={mse_test:.6f}, EVR={evr_test:.4f}')
        results_ae_test[dim_key] = build_result_entry(
            latent_factors=latent_test,
            reconstructed=reconstructed_test,
            mse=mse_test,
            evr=evr_test,
        )

        latent_full, _, _, evr_full, reconstructed_full, _ = model.evaluate_on_data(X_scaled)
        mse_full = reconstruction_mse(X_scaled, reconstructed_full)
        print(f'Autoencoder full data -> MSE={mse_full:.6f}, Total EVR={evr_full:.4f}')
        results_ae_full[dim_key] = build_result_entry(
            latent_factors=latent_full,
            reconstructed=reconstructed_full,
            mse=mse_full,
            evr=evr_full,
        )

    return results_ae_test, results_ae_full


results_ae_test, results_ae_full = run_autoencoder_analysis(
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    X_scaled=X_scaled,
    dimensions=DIMENSIONS,
    seed=SEED,
)


## 8. Results dictionary and comparison table

In [ ]:

analysis_results = build_analysis_results()
analysis_results['EFA']['results'] = results_efa
analysis_results['PCA']['results'] = results_pca
analysis_results['PCA_outer_cv']['results'] = results_pca_cv
analysis_results['KPCA_linear']['results'] = results_kpca_linear
analysis_results['KPCA_rbf_nested_cv']['results'] = results_kpca_rbf_cv
analysis_results['Autoencoder']['results'] = results_ae_full
analysis_results['Autoencoder_test']['results'] = results_ae_test

comparison_table = build_comparison_table(analysis_results, DIMENSIONS)
display(comparison_table)


## 9. Save latent factors and summary tables

In [ ]:

# Save full-data latent factors.
save_latent_factors(OUTPUT_DIR, 'efa', results_efa, DIMENSIONS)
save_latent_factors(OUTPUT_DIR, 'pca', results_pca, DIMENSIONS)
save_latent_factors(OUTPUT_DIR, 'kpca_linear', results_kpca_linear, DIMENSIONS)
save_latent_factors(OUTPUT_DIR, 'autoencoder', results_ae_full, DIMENSIONS)

comparison_table.to_csv(OUTPUT_DIR / 'comparison_table_long.csv', index=False)

comparison_table_wide = comparison_table.pivot_table(
    index='dim',
    columns=['analysis_key', 'evaluation'],
    values=['mse', 'evr', 'evr_std'],
    aggfunc='first',
)
comparison_table_wide.to_csv(OUTPUT_DIR / 'comparison_table_wide.csv')

# Full-data summary table.
full_data_summary = comparison_table[comparison_table['evaluation'] == 'full_data_reconstruction'].copy()
full_data_summary.to_csv(OUTPUT_DIR / 'comparison_full_data_only.csv', index=False)

# Cross-validated summary table.
cv_summary = comparison_table[comparison_table['evaluation'].isin(['outer_cv_reconstruction_evr', 'nested_cv_reconstruction_evr'])].copy()
cv_summary.to_csv(OUTPUT_DIR / 'comparison_cross_validated_only.csv', index=False)

autoencoder_test_summary = comparison_table[comparison_table['evaluation'] == 'test_set_reconstruction'].copy()
autoencoder_test_summary.to_csv(OUTPUT_DIR / 'autoencoder_test_set_summary.csv', index=False)

print('Saved summary tables:')
print(OUTPUT_DIR / 'comparison_table_long.csv')
print(OUTPUT_DIR / 'comparison_table_wide.csv')
print(OUTPUT_DIR / 'comparison_full_data_only.csv')
print(OUTPUT_DIR / 'comparison_cross_validated_only.csv')
print(OUTPUT_DIR / 'autoencoder_test_set_summary.csv')


## 10. Plot and save figures

In [ ]:

# EVR across analysis variants.
plot_metric_line(
    comparison_df=comparison_table,
    metric='evr',
    title='Total Explained Variance vs Encoding Dimension',
    ylabel='Total Explained Variance',
    output_file=OUTPUT_DIR / 'evr_vs_dimension_all_variants.png',
    allowed_evaluations=['full_data_reconstruction', 'outer_cv_reconstruction_evr', 'nested_cv_reconstruction_evr'],
)

# Reconstruction MSE for analyses with reconstruction outputs.
plot_metric_line(
    comparison_df=comparison_table,
    metric='mse',
    title='Reconstruction MSE vs Encoding Dimension',
    ylabel='Reconstruction MSE',
    output_file=OUTPUT_DIR / 'mse_vs_dimension_full_data_and_test.png',
    allowed_evaluations=['full_data_reconstruction', 'test_set_reconstruction'],
)

# Grouped-bar EVR view.
plot_evr_grouped_bar(
    comparison_df=comparison_table,
    output_file=OUTPUT_DIR / 'evr_grouped_bar.png',
    allowed_evaluations=['full_data_reconstruction', 'outer_cv_reconstruction_evr', 'nested_cv_reconstruction_evr'],
)


## 11. Output summary

In [ ]:

print(f'All outputs were saved to: {OUTPUT_DIR.resolve()}')
print('\nFull-data comparison across EFA, PCA, KPCA (linear), and Autoencoder:')
display(full_data_summary)

print('\nCross-validated comparison results retained from the original notebook logic:')
display(cv_summary)

print('\nAutoencoder held-out test-set results:')
display(autoencoder_test_summary)
